# 1.安装依赖与初始化大模型

In [13]:
# 安装完整依赖包
!pip install -q langchain-openai langgraph chromadb tabulate pydantic matplotlib

import os
import re
import time
import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import TypedDict, List, Dict, Any
from IPython.display import display, Markdown
from kaggle_secrets import UserSecretsClient
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage ,AIMessage
from langgraph.graph import StateGraph, START, END

# ==============================================================================
# 调用大模型 API 
# ==============================================================================
class ResilientFreeLLMRouter:
    def __init__(self):
        self.user_secrets = UserSecretsClient()
        self.providers = []
        self.current_idx = 0
        self._init_free_providers()
        
    def _get_secret_safe(self, key_name: str) -> str:
        try:
            val = self.user_secrets.get_secret(key_name)
            return val.strip() if val else ""
        except Exception:
            return ""

    def _init_free_providers(self):
        candidate_configs = [
            # 1. Google
            {
                "name": "Google-Gemini-4-hydrogen",
                "secret_key": "GEMINI API KEY",
                "model": "gemini-4-hydrogen",
                "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                "timeout": 30,
                "min_interval": 2.0
            },
            {
                "name": "Google-Gemini-3.8-Flash",
                "secret_key": "GEMINI API KEY",
                "model": "gemini-3.8-flash",
                "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                "timeout": 30,
                "min_interval": 2.0
            },
            {
                "name": "Google-Gemini-3.5-Flash",
                "secret_key": "GEMINI API KEY",
                "model": "gemini-3.5-flash-lite",
                "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
                "timeout": 30,
                "min_interval": 2.0
            },
            # 2. 阿里魔搭 ModelScope
            {
                "name": "ModelScope-Qwen3.5-35B",
                "secret_key": "MODELSCOPE_API_KEY",
                "model": "Qwen/Qwen3.5-35B-A3B",
                "base_url": "https://api-inference.modelscope.cn/v1/",
                "timeout": 120,
                "min_interval": 2.0
            },
            {
                "name": "ModelScope-Qwen3.8-27B",
                "secret_key": "MODELSCOPE_API_KEY",
                "model": "Qwen/Qwen3.8-27B",
                "base_url": "https://api-inference.modelscope.cn/v1/",
                "timeout": 120,
                "min_interval": 2.0
            },
            # 3. 智谱 BigModel
            {
                "name": "Zhipu-GLM4-Flash-250414",
                "secret_key": "ZHIPU_API_KEY",
                "model": "glm-4-flash-250414",
                "base_url": "https://open.bigmodel.cn/api/paas/v4/",
                "timeout": 90,
                "min_interval": 3.0
            },
            {
                "name": "Zhipu-GLM4-Flash",
                "secret_key": "ZHIPU_API_KEY",
                "model": "glm-4-flash",
                "base_url": "https://open.bigmodel.cn/api/paas/v4/",
                "timeout": 90,
                "min_interval": 3.0
            }
        ]
        
        for cfg in candidate_configs:
            api_key = self._get_secret_safe(cfg["secret_key"])
            if api_key:
                client = ChatOpenAI(
                    model=cfg["model"],
                    api_key=api_key,
                    base_url=cfg["base_url"],
                    temperature=0.0,
                    max_retries=0,      # 设为 0！防止底层 SDK 暗中重试撞上智谱 1 并发红线
                    timeout=cfg["timeout"]
                )
                self.providers.append({
                    "name": cfg["name"],
                    "client": client,
                    "min_interval": cfg["min_interval"],
                    "cooldown_until": 0.0,
                    "last_call_time": 0.0,
                    "success_cnt": 0,
                    "fail_cnt": 0
                })
                
        assert len(self.providers) > 0, "未检测到任何 API Key，请检查 Kaggle Secrets！"
        print(f"检测到有 {len(self.providers)} 条独立模型通道。")

    def invoke(self, input_data: Any, max_total_attempts: int = 12) -> Any:
        n_providers = len(self.providers)
        last_error = None
        
        for attempt in range(1, max_total_attempts + 1):
            now = time.time()
            selected_provider = None
            
            # Round-Robin 寻找未冷却的通道
            for _ in range(n_providers):
                p = self.providers[self.current_idx]
                self.current_idx = (self.current_idx + 1) % n_providers
                if now >= p["cooldown_until"]:
                    selected_provider = p
                    break
            
            # 若全在冷却，等待最快恢复的一条
            if selected_provider is None:
                earliest = min(self.providers, key=lambda x: x["cooldown_until"])
                wait_sec = max(1.0, earliest["cooldown_until"] - now)
                print(f"所有通道都在短暂冷却，等待 {wait_sec:.1f}s 后由 [{earliest['name']}] 恢复...")
                time.sleep(wait_sec)
                selected_provider = earliest
            
            # 严格遵守单通道安全间隔（防止智谱 1 并发碰撞）
            elapsed = time.time() - selected_provider["last_call_time"]
            if elapsed < selected_provider["min_interval"]:
                time.sleep(selected_provider["min_interval"] - elapsed)
                
            try:
                selected_provider["last_call_time"] = time.time()
                response = selected_provider["client"].invoke(input_data)
                selected_provider["success_cnt"] += 1
                return response
                
            except Exception as e:
                last_error = e
                err_str = str(e).lower()
                selected_provider["fail_cnt"] += 1
                
                if "404" in err_str or "not found" in err_str or "does not exist" in err_str or "400" in err_str:
                    # 模型名无效/暂不支持，直接长冷却 10 分钟跳过
                    selected_provider["cooldown_until"] = time.time() + 600.0
                    print(f"  通道 [{selected_provider['name']}] 模型不可用 -> 自动屏蔽并切下一通道")
                elif "429" in err_str or "rate" in err_str or "limit" in err_str or "1302" in err_str:
                    # 429 限流 -> 冷却 35 秒
                    selected_provider["cooldown_until"] = time.time() + 35.0
                    print(f"  通道 [{selected_provider['name']}] 限流 (429) -> 冷却 35s，秒切下一通道 ({attempt}/{max_total_attempts})")
                elif "503" in err_str or "high demand" in err_str or "overloaded" in err_str:
                    # 503 平台拥堵 -> 冷却 25 秒
                    selected_provider["cooldown_until"] = time.time() + 25.0
                    print(f"  通道 [{selected_provider['name']}] 服务端拥堵 (503) -> 冷却 25s，切下一通道 ({attempt}/{max_total_attempts})")
                else:
                    # 网络超时 -> 短冷却 10 秒
                    selected_provider["cooldown_until"] = time.time() + 120.0
                    print(f"  通道 [{selected_provider['name']}] 异常 ({str(e)[:45]}...) -> 切下一通道 ({attempt}/{max_total_attempts})")
                time.sleep(1.0)
                
        raise RuntimeError(f"所有通道重试 {max_total_attempts} 次后仍失败，最后报错: {last_error}")

    def diagnose_providers(self):
            """逐个测试每条通道的真实连通性与响应速度，并对故障通道自动预设冷却期"""
            print("正在对全部可用模型通道进行连通性与耗时体检...\n")
            for p in self.providers:
                t0 = time.time()
                try:
                    p["last_call_time"] = time.time()
                    res = p["client"].invoke("只回复两个字：正常")
                    cost = time.time() - t0
                    print(f"  [{p['name']:<36}] 连通! 耗时: {cost:.2f}s | 状态: {res.content.strip()[:15]}")
                except Exception as e:
                    cost = time.time() - t0
                    err_str = str(e).lower()
                    
                    # 分级预设冷却期，避免后续正式任务刚启动就反复撞死通道
                    if any(k in err_str for k in ["404", "not found", "does not exist", "400"]):
                        cd, tag = 600.0, "模型不可用，屏蔽 600s"
                    elif any(k in err_str for k in ["429", "rate", "limit", "quota", "1302"]):
                        cd, tag = 60.0, "配额/限流(429)，冷却 60s"
                    elif any(k in err_str for k in ["timed out", "timeout"]):
                        cd, tag = 120.0, "请求超时，熔断冷却 120s"
                    elif any(k in err_str for k in ["503", "high demand", "overloaded"]):
                        cd, tag = 45.0, "服务拥堵(503)，冷却 45s"
                    else:
                        cd, tag = 30.0, "连接异常，冷却 30s"
                        
                    p["cooldown_until"] = time.time() + cd
                    print(f"  [{p['name']:<36}] 失败! 耗时: {cost:.2f}s | {tag} | 原因: {str(e)[:55]}")
                time.sleep(1.5)

    def print_stats(self):
        stats = [{
            "通道名称": p["name"],
            "成功次数": p["success_cnt"],
            "异常切换": p["fail_cnt"],
            "状态": "正常" if time.time() >= p["cooldown_until"] else "冷却中"
        } for p in self.providers]
        display(Markdown(pd.DataFrame(stats).to_markdown(index=False)))

# 实例化
llm = ResilientFreeLLMRouter()
llm.diagnose_providers()

检测到有 7 条独立模型通道。
正在对全部可用模型通道进行连通性与耗时体检...

  [Google-Gemini-4-hydrogen            ] 失败! 耗时: 30.05s | 请求超时，熔断冷却 120s | 原因: Request timed out.
  [Google-Gemini-3.8-Flash             ] 连通! 耗时: 18.27s | 状态: 正常
  [Google-Gemini-3.5-Flash             ] 连通! 耗时: 0.60s | 状态: 正常
  [ModelScope-Qwen3.5-35B              ] 连通! 耗时: 4.76s | 状态: 正常
  [ModelScope-Qwen3.8-27B              ] 连通! 耗时: 3.33s | 状态: 正常
  [Zhipu-GLM4-Flash-250414             ] 连通! 耗时: 2.39s | 状态: 正常
  [Zhipu-GLM4-Flash                    ] 连通! 耗时: 1.11s | 状态: 正常


# 2.构建全量本地数仓

In [14]:
# 1. 自动检索 Kaggle 挂载的 Olist 数据集路径
data_dir = ""
for root, dirs, files in os.walk("/kaggle/input"):
    if "olist_orders_dataset.csv" in files:
        data_dir = root
        break

assert data_dir != "", "未找到数据集，请在右侧 Input 面板挂载 Brazilian E-Commerce Public Dataset！"

# 2. 创建 SQLite 数据库并导入全部 8 张核心表
db_path = "/kaggle/working/olist_enterprise.db"
conn = sqlite3.connect(db_path)

tables_map = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "sellers": "olist_sellers_dataset.csv"
}

print("正在构建企业级模拟数仓...")
for table_name, file_name in tables_map.items():
    df = pd.read_csv(os.path.join(data_dir, file_name))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"  ├─ 表 [{table_name:<20}] 导入成功 | 行数: {len(df):<8} | 核心字段: {list(df.columns[:4])}...")
print("数仓构建完毕！")

正在构建企业级模拟数仓...
  ├─ 表 [orders              ] 导入成功 | 行数: 99441    | 核心字段: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']...
  ├─ 表 [order_items         ] 导入成功 | 行数: 112650   | 核心字段: ['order_id', 'order_item_id', 'product_id', 'seller_id']...
  ├─ 表 [order_payments      ] 导入成功 | 行数: 103886   | 核心字段: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments']...
  ├─ 表 [customers           ] 导入成功 | 行数: 99441    | 核心字段: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city']...
  ├─ 表 [products            ] 导入成功 | 行数: 32951    | 核心字段: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght']...
  ├─ 表 [category_translation] 导入成功 | 行数: 71       | 核心字段: ['product_category_name', 'product_category_name_english']...
  ├─ 表 [order_reviews       ] 导入成功 | 行数: 99224    | 核心字段: ['review_id', 'order_id', 'review_score', 'review_comment_title']...
  ├─ 表 [sellers             ] 导入成功 | 行数: 3095     | 核

# 3.构建BI知识库

除了把业务知识注入外，还加上了检索功能，为什么需要进行检索，而不直接每次都查全部知识库信息就行？
1. 真实业务场景往往数据表很多，如果每次都查询全部信息，会有：
   - 大模型响应速度变慢
   - 超出大模型上下文窗口上限
   - 消耗更多的token，成本上涨
2. 如果某些表信息接近，可能稀释大模型注意力，选错相近字段从而得到错误的结果

In [15]:
import chromadb

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("enterprise_bi_kb")
except:
    pass
collection = chroma_client.create_collection(name="enterprise_bi_kb")

# 1. 数仓表结构与主外键关联规范 
schema_docs = [
    "【表结构：orders (订单主表)】主键: order_id | 关联键: customer_id | 字段: order_status(状态: delivered已送达, canceled已取消, shipped已发货等), order_purchase_timestamp(下单时间), order_approved_at(支付审批时间), order_delivered_customer_date(实际送达客户时间), order_estimated_delivery_date(预计送达时间)。",
    "【表结构：order_items (订单商品明细表)】联合主键: (order_id, order_item_id) | 字段: product_id(商品ID), seller_id(卖家ID), price(单件商品售价，不含运费), freight_value(单件商品运费)。注意：一笔订单买N件商品就会有N行记录！",
    "【表结构：order_payments (订单支付明细表)】联合主键: (order_id, payment_sequential) | 字段: payment_type(支付方式: credit_card, boleto, voucher, debit_card), payment_installments(分期数), payment_value(该笔支付金额)。注意：一笔订单可能有多次支付记录（多行）！",
    "【表结构：customers (客户表)】主键: customer_id(单笔订单绑定的临时客户ID) | 核心字段: customer_unique_id(客户真实唯一标识ID，同一个人在不同订单中 customer_unique_id 相同，但 customer_id 不同！), customer_city, customer_state(州缩写，如 'SP', 'RJ', 'MG')。",
    "【表结构：products & category_translation (商品与英文品类映射表)】products表主键: product_id，含葡萄牙语品类名 product_category_name；必须通过 LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name 获取英文品类名 product_category_name_english。",
    "【表结构：order_reviews (订单评价表)】字段: review_id, order_id, review_score(评分1到5分)。注意：同一个 order_id 可能存在多条评价记录。"
]

# 2. 定义指标口径与注意事项
metric_docs = [
    "【口径1：有效订单与有效GMV (Net Valid GMV)】除非特别说明，所有‘有效订单’、‘有效GMV’、‘销售额’均要求过滤 orders.order_status = 'delivered'。有效GMV = SUM(order_items.price)，即仅统计商品售价总和，不含运费(freight_value)。",
    "【口径2：独立买家数 (UV)】统计买家人数、独立客户数或UV时，必须关联 customers 表并使用 COUNT(DISTINCT customers.customer_unique_id)，严禁使用 COUNT(DISTINCT customer_id)！",
    "【口径3：客单价 (AOV) 与 件单价 (ASP) 区分】有效客单价 (AOV) = SUM(order_items.price) / COUNT(DISTINCT orders.order_id)，分母必须是去重后的有效订单数；有效件单价 (ASP) = SUM(order_items.price) / COUNT(order_items.order_item_id)，分母是商品件数（明细行数）。",
    "【口径4：订单连带率 (UPT / Attach Rate)】指平均每笔有效订单包含的商品件数，计算公式为：COUNT(order_items.order_item_id) * 1.0 / COUNT(DISTINCT orders.order_id)（需限定 order_status = 'delivered'）。",
    "【口径5：订单取消率 (Order Cancellation Rate)】在指定下单时间范围内所有订单中，状态为 canceled 的订单占比，公式：SUM(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(DISTINCT order_id)。注意：算取消率时 WHERE 条件不能过滤 order_status = 'delivered'！",
    "【口径6：平均履约时长 (Avg Delivery Days) 与 履约延迟率 (Delay Rate)】均限定在 order_status = 'delivered' 且 order_delivered_customer_date IS NOT NULL 的订单中。平均履约时长(天) = AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp))；履约延迟率(%) = AVG(CASE WHEN julianday(order_delivered_customer_date) > julianday(order_estimated_delivery_date) THEN 1.0 ELSE 0.0 END) * 100.0。",
    "【口径7：有效订单差评率 (Bad Review Rate)】在已送达(delivered)且关联到评价表(order_reviews)的订单中，评分 <= 2 分的去重订单数除以总参评去重有效订单数，公式：COUNT(DISTINCT CASE WHEN r.review_score <= 2 THEN o.order_id END) * 100.0 / COUNT(DISTINCT o.order_id)。必须用 INNER JOIN order_reviews r ON o.order_id = r.order_id。",
    "【口径8：防扇出膨胀规范（一对多关联防笛卡尔积 Fan-out Trap）】由于 order_items（一单多品）和 order_payments（一单多次支付）或 order_reviews 同属订单的下游一对多明细表，当需要按支付方式（如 credit_card）筛选并统计商品有效GMV（SUM(oi.price)）时，严禁直接将 order_items 与 order_payments 直接 JOIN 后求和！必须使用子查询过滤：WHERE o.order_id IN (SELECT DISTINCT order_id FROM order_payments WHERE payment_type = 'credit_card')。",
    "【口径9：周期用户复购率 (Repeat Purchase Rate)】在指定统计周期内产生过有效订单(delivered)的独立买家中，有效订单数 >= 2 笔的独立买家占比(%)。必须先用 CTE 按 customer_unique_id 聚合计算 order_cnt = COUNT(DISTINCT o.order_id)，再用 SUM(CASE WHEN order_cnt >= 2 THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(customer_unique_id)。",
    "【口径10：新老客界定口径 (New vs Returning Customer)】以真实用户 customer_unique_id 在全量历史有效订单(delivered)中的首次下单时间 min_purchase_time = MIN(order_purchase_timestamp) 为准。若某用户在全量历史中的首次有效下单时间早于目标统计周期的起始时间（例如分析2018-08时，其首次有效下单时间 < '2018-08-01'），则该用户在目标周期内属于【老客】；否则属于【新客】。"
]

# 3. 复杂SQL参考样本
golden_sql_docs = [
    """【参考范式1：防扇出膨胀（统计特定支付方式下的有效GMV）】
    SELECT ROUND(SUM(oi.price), 2) AS val
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
      AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08'
      AND o.order_id IN (SELECT DISTINCT order_id FROM order_payments WHERE payment_type = 'credit_card');""",

    """【参考范式2：使用 CTE 计算周期用户复购率(%)】
    WITH user_orders AS (
        SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS ord_cnt
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
          AND strftime('%Y', o.order_purchase_timestamp) = '2018'
        GROUP BY c.customer_unique_id
    )
    SELECT ROUND(SUM(CASE WHEN ord_cnt >= 2 THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(customer_unique_id), 2) AS val
    FROM user_orders;""",

    """【参考范式3：使用 CTE 界定历史老客并统计其在目标月份贡献的有效GMV】
    WITH user_first_order AS (
        SELECT c.customer_unique_id, MIN(o.order_purchase_timestamp) AS first_order_time
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id
    )
    SELECT ROUND(SUM(oi.price), 2) AS val
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN user_first_order ufo ON c.customer_unique_id = ufo.customer_unique_id
    WHERE o.order_status = 'delivered'
      AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08'
      AND ufo.first_order_time < '2018-08-01';"""
]

all_kb_docs = schema_docs + metric_docs + golden_sql_docs
collection.add(
    documents=all_kb_docs,
    ids=[f"kb_{i}" for i in range(len(all_kb_docs))]
)

# 语义检索
def retrieve_bi_context(query: str, top_k: int = 8) -> str:
    results = collection.query(query_texts=[query], n_results=min(top_k, len(all_kb_docs)))
    return "\n\n".join(results["documents"][0])

print(f"BI语义知识库构建完成！共载入 {len(all_kb_docs)} 条 Schema、口径规范与范式样本。")

BI语义知识库构建完成！共载入 19 条 Schema、口径规范与范式样本。


# 4.Text-to-SQL 自纠错 Agent

In [17]:
import time

def call_llm_with_rate_limit_guard(messages, sleep_seconds: float = 2.0, api_retries: int = 3) -> str:
    """
    安全调用大模型 API 的封装函数：
    1. 每次调用前固定主动休眠 sleep_seconds 秒，避免频繁调度触发并发限制
    2. 若遇到 429 Rate Limit 或网络波动，自动执行指数退避等待并重调接口
    """
    if sleep_seconds > 0:
        time.sleep(sleep_seconds)
        
    for i in range(1, api_retries + 1):
        try:
            return llm.invoke(messages).content
        except Exception as api_err:
            wait_time = sleep_seconds * (i * 3)  # 依次等待 6秒、12秒、18秒
            print(f"[API 限流保护] 接口请求受限 ({str(api_err)[:50]}...)，正在休眠 {wait_time} 秒后重试 ({i}/{api_retries})...")
            time.sleep(wait_time)
    raise RuntimeError("大模型 API 连续请求失败，请检查账户额度或稍后再试。")


def execute_sql_with_self_correction(
    task_description: str, 
    max_retries: int = 3, 
    sleep_seconds: float = 2.0,  # <--- 新增：每次请求 API 前的休眠秒数（默认2秒）
    verbose: bool = True
) -> Dict[str, Any]:
    """
    核心 Text-to-SQL 引擎：
    1. 动态检索业务口径与 Few-shot SQL
    2. 每次请求大模型前自动休眠 sleep_seconds 秒
    3. 沙盒执行 SQL，捕获语法报错、空表异常、全 Null 异常并自动重写
    """
    bi_context = retrieve_bi_context(task_description, top_k=9)
    
    system_prompt = f"""你是一名精通 SQLite 语法的资深电商 BI 架构师。
请严格依据下方提供的【数仓表结构、10大业务指标口径规范与参考SQL范式】，为用户编写 SQLite 查询语句。

【要求】：
1. 严格遵循业务口径（如有效GMV必须过滤 order_status = 'delivered' 且只算 price；算UV必须用 customer_unique_id；关联支付表算GMV必须防笛卡尔积膨胀）。
2. 时间过滤优先使用 strftime('%Y-%m', col) 或字符串比较（如 col < '2018-08-01'），时间差计算使用 julianday(col1) - julianday(col2)。
3. 只输出纯 SQL 代码，用 ```sql 和 ``` 包裹，严禁输出任何多余文字。

【BI 语义库上下文】：
{bi_context}
"""
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"查询任务：{task_description}")
    ]
    
    sql = ""
    err_msg = ""
    for attempt in range(1, max_retries + 1):
        # 使用带休眠与防限流保护的函数调用大模型
        resp = call_llm_with_rate_limit_guard(messages, sleep_seconds=sleep_seconds)
        sql = resp.split("```sql")[-1].split("```")[0].strip() if "```sql" in resp else resp.strip()
        
        try:
            df = pd.read_sql(sql, conn)
            # 异常校验 1：返回 0 行
            if df.empty:
                raise ValueError("查询结果为空表（0行）！请检查时间范围格式或枚举值条件是否过于严苛。")
            # 异常校验 2：首行全为 NULL
            if df.iloc[0].isna().all():
                raise ValueError("查询结果全为 NULL！请检查过滤条件是否导致没有匹配行。")
                
            if verbose:
                print(f" [SQL Agent] 第 {attempt} 次执行成功 (已休眠 {sleep_seconds}s | 返回 {len(df)} 行)")
            return {"status": "success", "sql": sql, "data": df.to_markdown(index=False), "df": df, "attempts": attempt, "error": None}
            
        except Exception as e:
            err_msg = str(e)
            if verbose:
                print(f" [SQL Agent] 第 {attempt} 次异常 ({err_msg[:60]}...) -> 休眠 {sleep_seconds}s 后触发自纠错重写")
            messages.append(SystemMessage(content=resp))
            AIMessage(content=resp)
            messages.append(HumanMessage(
                content=f"执行报错或结果异常：【{err_msg}】。\n请仔细对照【BI 语义库上下文】中的字段名、SQLite 时间函数和口径规范修正 SQL，仅输出修正后的 ```sql 代码块。"
            ))
            
    return {"status": "failed", "sql": sql, "data": "查询失败", "df": pd.DataFrame(), "attempts": max_retries, "error": err_msg}

print("Text-to-SQL自纠错引擎已更新！")

Text-to-SQL自纠错引擎已更新！


# 5.基于 LangGraph 的多 Agent 异动归因

定义4个Agent角色：
1. Planner（规划Agent）：当用户问“为什么2018年8月销售额环比下降了？”时，Planner 不直接写SQL，而是结合业务知识拆解分析假设（假设1：某核心品类下跌；假设2：某重点城市物流延迟导致取消率上升）
2. SQL Executor（取数Agent）：并行或串行调用Text-to-SQL 模块，拉取不同维度的拆解数据
3. Analyst（分析Agent）：获取数据后，编写代码计算各维度的贡献度（Contribution Rate）或相关系数，并生成可视化图表保存
4. Reporter（报告Agent）：汇总上述图表和数据结论，输出结构化的归因分析报告

In [18]:
# 定义多 Agent 状态图
class AgentState(TypedDict):
    user_question: str
    analysis_plan: List[str]
    queried_results: List[Dict]
    quant_summary: str
    final_report: str

def planner_node(state: AgentState):
    print("\n[1. Planner Agent] 正在按电商归因框架拆解子任务...")
    prompt = f"""
    你是资深数据分析师。针对业务问题：【{state['user_question']}】
    请从以下 3 个正交维度拆解为 3 条具体的数据查询指令：
    1. 大盘核心指标对比（2018-07 vs 2018-08 的有效GMV、有效订单数、有效客单价AOV、独立买家UV、履约延迟率）。
    2. 区域维度下钻（对比 2018-07 vs 2018-08 有效GMV排名前5的客户所在州 customer_state 的有效GMV及订单数变化）。
    3. 品类维度下钻（对比 2018-07 vs 2018-08 有效GMV排名前5的英文商品品类 product_category_name_english 的有效GMV变化）。
    需要遵循：
    1. 请严格只输出包含 3 个字符串的 JSON 数组：["指令1", "指令2", "指令3"]
    2. 严禁输出任何 SQL 代码或字段名！只输出 3 条纯中文业务分析指令”
"""
    resp = llm.invoke(prompt).content
    clean_json = resp.split("```json")[-1].split("```")[0].strip() if "```json" in resp else resp.strip()
    plan = json.loads(clean_json)
    return {"analysis_plan": plan}

def sql_executor_node(state: AgentState):
    print("\n[2. SQL Executor Agent] 正在调用自纠错 SQL 引擎拉取多维数据...")
    results = []
    for idx, task in enumerate(state["analysis_plan"], 1):
        print(f" ├─ 子任务 {idx}: {task[:50]}...")
        res = execute_sql_with_self_correction(task, verbose=True)
        results.append({"task": task, "sql": res["sql"], "markdown_data": res["data"]})
    return {"queried_results": results}

def python_analyst_node(state: AgentState):
    print("\n[3. Python Analyst Agent] 正在计算环比增速与异动贡献度 (Contribution Rate)...")
    raw_data_str = "\n\n".join([f"### 任务：{r['task']}\n{r['markdown_data']}" for r in state["queried_results"]])
    prompt = f"""你是量化归因分析师。请基于下方真实查询数据，对【{state['user_question']}】做精确量化计算：
1. 大盘拆解：计算有效GMV、订单量、客单价AOV、延迟率的差值(Delta)与环比增速(MoM%)，运用 LMDI/量价拆解思想判断大盘变动主要由【订单量】还是【客单价】驱动。
2. 贡献度拆解：计算各省份(State)与各品类(Category)对总 GMV 变动的【异动贡献度】（某维度GMV变化量 / 大盘总GMV变化量 * 100%），找出最大的拖累项top5与拉动项top5。
【真实数据】：\n{raw_data_str}"""
    return {"quant_summary": llm.invoke(prompt).content}

def reporter_node(state: AgentState):
    print("\n[4. Reporter Agent] 正在生成结构化商业诊断报告...")
    prompt = f"""你是首席BI分析师，请基于以下量化计算结果撰写《2018年8月电商大盘异动归因分析报告》（Markdown格式，含核心结论、量价拆解表、区域与品类贡献度Top榜、业务策略建议）：
                \n{state['quant_summary']}。
                【格式硬性要求】：
                1. 直接输出纯正的 Markdown 正文内容，严禁在最外层使用 ```markdown 或 ``` 代码块包裹整篇报告！
                2. 对于表格类的输出，请检查正确格式后进行输出，可以直观阅读表格而非文字+符号的排版
            """
    return {"final_report": llm.invoke(prompt).content}

workflow = StateGraph(AgentState)
workflow.add_node("planner", planner_node)
workflow.add_node("sql_executor", sql_executor_node)
workflow.add_node("python_analyst", python_analyst_node)
workflow.add_node("reporter", reporter_node)
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "sql_executor")
workflow.add_edge("sql_executor", "python_analyst")
workflow.add_edge("python_analyst", "reporter")
workflow.add_edge("reporter", END)
attribution_app = workflow.compile()

# 执行一次完整的异动归因分析
user_q = "对比 2018年7月与 2018年8月，分析有效GMV变动的原因，并从量价漏斗、省份区域、商品品类维度计算异动贡献度。"
final_output = attribution_app.invoke({"user_question": user_q})
display(Markdown(final_output["final_report"]))


[1. Planner Agent] 正在按电商归因框架拆解子任务...
  通道 [Google-Gemini-4-hydrogen] 限流 (429) -> 冷却 35s，秒切下一通道 (1/12)

[2. SQL Executor Agent] 正在调用自纠错 SQL 引擎拉取多维数据...
 ├─ 子任务 1: 查询2018年7月与2018年8月大盘的有效成交金额、有效订单量、客单价、独立购买买家数及订单履约延...
 [SQL Agent] 第 1 次执行成功 (已休眠 2.0s | 返回 2 行)
 ├─ 子任务 2: 查询2018年7月与2018年8月有效成交金额排名前5位的客户所在省份或州，对比其有效成交金额与订单...
  通道 [ModelScope-Qwen3.5-35B] 异常 (Request timed out....) -> 切下一通道 (1/12)
  通道 [ModelScope-Qwen3.8-27B] 异常 (<html>
<head><title>502 Bad Gateway</title><...) -> 切下一通道 (2/12)
 [SQL Agent] 第 1 次异常 (Execution failed on sql 'WITH ValidGMV AS (
    SELECT
     ...) -> 休眠 2.0s 后触发自纠错重写
 [SQL Agent] 第 2 次异常 (Execution failed on sql 'WITH ValidGMV AS (
    SELECT
     ...) -> 休眠 2.0s 后触发自纠错重写
  通道 [Google-Gemini-4-hydrogen] 服务端拥堵 (503) -> 冷却 25s，切下一通道 (1/12)
  通道 [Google-Gemini-3.8-Flash] 异常 (Request timed out....) -> 切下一通道 (2/12)
 [SQL Agent] 第 3 次异常 (Execution failed on sql 'WITH ValidGMV AS (
    SELECT
     ...) -> 休眠 2.0s 后触发自纠错重写
 ├─ 子任务 3: 查询2018年7月与2018年8月有效成交金额排名前5位的主

# 2018年8月电商大盘异动归因分析报告

## 核心结论

2018年8月，电商大盘GMV出现下降，主要原因是客单价下降，尽管订单量有所增长。具体来说，客单价下降对GMV的负面影响远大于订单量增长带来的正面影响。

## 量价拆解表

| 指标             | 2018-07 | 2018-08 | Delta | MoM%   |
|------------------|---------|---------|-------|--------|
| 有效GMV          | 867953  | 838577  | -29376 | -3.38% |
| 有效订单量       | 6159    | 6351    | 192   | 3.12%  |
| 客单价(AOV)      | 140.92  | 132.04  | -8.88 | -6.31% |
| 独立购买买家数(UV)| 6100    | 6310    | 210   | 3.44%  |
| 订单履约延迟率   | 4.37%   | 10.35%  | 5.98% | -     |

## 区域与品类贡献度Top榜

### 省份贡献度

由于省份数据缺失，无法提供具体省份贡献度分析。

### 品类贡献度

| 品类              | GMV_Jul | GMV_Aug | GMV_Change | MoM_Growth% | Contribution_to_Market_Change |
|-------------------|---------|---------|------------|-------------|------------------------------|
| beleza_saude      | 103524  | 119391  | 15967      | 15.33%      | -54.01%                      |
| relogios_presentes| 95164.8 | 69767.1 | -25397.7   | -26.69%     | 86.46%                       |
| cama_mesa_banho   | 54405.2 | 60891.1 | 6485.9     | 11.92%      | -22.08%                      |
| utilidades_domesticas | 62583.3 | 58763.4 | -3819.9    | -6.10%      | 13.00%                       |
| esporte_lazer     | 54015.9 | 50860.2 | -3155.7    | -5.84%      | 10.74%                       |
| **Total**         | **319880** | **302520** | **-17460** |             | **100.00%**                  |

**Top 5 拖累项（贡献度最高负值）**：
1. relogios_presentes: -86.46%
2. beleza_saude: -54.01%
3. cama_mesa_banho: -22.08%
4. esporte_lazer: -10.74%
5. utilidades_domesticas: -13.00%

**Top 5 拉动项（贡献度最高正值）**：
1. utilidades_domesticas: 13.00%
2. esporte_lazer: 10.74%
3. cama_mesa_banho: -22.08%
4. beleza_saude: -54.01%
5. relogios_presentes: 86.46%

## 业务策略建议

1. 深入分析relogios_presentes品类GMV下降的原因，如促销策略、竞争加剧等，并采取相应措施。
2. 补充省份数据以完善区域贡献度分析，针对不同区域制定差异化的营销策略。
3. 关注客单价下降的趋势，通过优化产品组合、提升服务质量等方式提高客单价。
4. 加强对高贡献度品类的支持，如utilidades_domesticas和esporte_lazer，以推动整体GMV增长。

In [19]:
# ==============================================================================
# 1. 定义 15 道分层评测集，简单-中等-困难
# ==============================================================================
stratified_benchmark = [
    # ---------------- 简单 (基础取数与单一业务口径对齐) ----------------
    {
        "id": "E1", "difficulty": "Easy", "trap_type": "有效GMV口径(剔除未送达与运费)",
        "question": "统计 2018年8月（2018-08）的总有效GMV（保留2位小数）。",
        "golden_sql": """
            SELECT ROUND(SUM(oi.price), 2) AS val
            FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered' AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08';
        """
    },
    {
        "id": "E2", "difficulty": "Easy", "trap_type": "独立买家UV口径(customer_unique_id)",
        "question": "统计 2018年8月（2018-08）产生有效订单的独立买家人数（UV）。",
        "golden_sql": """
            SELECT COUNT(DISTINCT c.customer_unique_id) AS val
            FROM orders o JOIN customers c ON o.customer_id = c.customer_id
            WHERE o.order_status = 'delivered' AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08';
        """
    },
    {
        "id": "E3", "difficulty": "Easy", "trap_type": "订单取消率口径(全量分母)",
        "question": "计算 2018年8月（2018-08）下单的订单取消率百分比（保留2位小数，如 0.85 表示 0.85%）。",
        "golden_sql": """
            SELECT ROUND(SUM(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(DISTINCT order_id), 2) AS val
            FROM orders WHERE strftime('%Y-%m', order_purchase_timestamp) = '2018-08';
        """
    },
    {
        "id": "E4", "difficulty": "Easy", "trap_type": "有效件单价ASP口径",
        "question": "计算 2018年7月（2018-07）有效订单的商品件单价 ASP（保留2位小数）。",
        "golden_sql": """
            SELECT ROUND(SUM(oi.price) / COUNT(oi.order_item_id), 2) AS val
            FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered' AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-07';
        """
    },
    {
        "id": "E5", "difficulty": "Easy", "trap_type": "平均履约时长(julianday语法)",
        "question": "计算 2018年7月（2018-07）有效订单的平均履约时长（单位：天，保留2位小数）。",
        "golden_sql": """
            SELECT ROUND(AVG(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)), 2) AS val
            FROM orders
            WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
              AND strftime('%Y-%m', order_purchase_timestamp) = '2018-07';
        """
    },

    # ---------------- 中等 (多表关联、复合比率与分母防错) ----------------
    {
        "id": "M1", "difficulty": "Medium", "trap_type": "客单价AOV分母防膨胀(去重订单数)",
        "question": "计算 2018年7月（2018-07）的有效客单价 AOV（保留2位小数）。",
        "golden_sql": """
            SELECT ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS val
            FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered' AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-07';
        """
    },
    {
        "id": "M2", "difficulty": "Medium", "trap_type": "订单连带率UPT口径",
        "question": "计算 2018年上半年（2018-01 至 2018-06）有效订单的订单连带率 UPT（保留4位小数）。",
        "golden_sql": """
            SELECT ROUND(COUNT(oi.order_item_id) * 1.0 / COUNT(DISTINCT o.order_id), 4) AS val
            FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered'
              AND strftime('%Y-%m', o.order_purchase_timestamp) BETWEEN '2018-01' AND '2018-06';
        """
    },
    {
        "id": "M3", "difficulty": "Medium", "trap_type": "四表关联与英文品类Top1有效GMV",
        "question": "查询 2018年8月（2018-08）有效GMV排名第1的英文商品品类，返回该品类的有效GMV金额（保留2位小数）。",
        "golden_sql": """
            SELECT ROUND(SUM(oi.price), 2) AS val
            FROM orders o
            JOIN order_items oi ON o.order_id = oi.order_id
            LEFT JOIN products p ON oi.product_id = p.product_id
            LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
            WHERE o.order_status = 'delivered' AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08'
            GROUP BY t.product_category_name_english
            ORDER BY val DESC LIMIT 1;
        """
    },
    {
        "id": "M4", "difficulty": "Medium", "trap_type": "履约延迟率口径",
        "question": "计算 2018年5月（2018-05）有效订单的履约延迟率百分比（保留2位小数）。",
        "golden_sql": """
            SELECT ROUND(AVG(CASE WHEN julianday(order_delivered_customer_date) > julianday(order_estimated_delivery_date) THEN 1.0 ELSE 0.0 END) * 100.0, 2) AS val
            FROM orders
            WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
              AND strftime('%Y-%m', order_purchase_timestamp) = '2018-05';
        """
    },
    {
        "id": "M5", "difficulty": "Medium", "trap_type": "参评有效订单差评率(订单级去重)",
        "question": "计算 2018年5月（2018-05）来自圣保罗州（customer_state = 'SP'）的有效订单差评率百分比（保留2位小数）。",
        "golden_sql": """
            SELECT ROUND(COUNT(DISTINCT CASE WHEN r.review_score <= 2 THEN o.order_id END) * 100.0 / COUNT(DISTINCT o.order_id), 2) AS val
            FROM orders o
            JOIN customers c ON o.customer_id = c.customer_id
            JOIN order_reviews r ON o.order_id = r.order_id
            WHERE o.order_status = 'delivered' AND c.customer_state = 'SP'
              AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-05';
        """
    },

    # ---------------- 困难 (防扇出膨胀、CTE复购、新老客历史归因、窗口占比) ----------------
    {
        "id": "H1", "difficulty": "Hard", "trap_type": "一对多关联防扇出膨胀(Fan-out Trap)",
        "question": "统计 2018年8月（2018-08）使用信用卡（payment_type = 'credit_card'）支付过的有效订单，其对应的商品有效GMV总和是多少（保留2位小数）？",
        "golden_sql": """
            SELECT ROUND(SUM(oi.price), 2) AS val
            FROM orders o
            JOIN order_items oi ON o.order_id = oi.order_id
            WHERE o.order_status = 'delivered'
              AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08'
              AND o.order_id IN (SELECT DISTINCT order_id FROM order_payments WHERE payment_type = 'credit_card');
        """
    },
    {
        "id": "H2", "difficulty": "Hard", "trap_type": "CTE二次聚合计算周期用户复购率",
        "question": "计算 2018年全年（2018-01 至 2018-12）产生过有效订单的独立买家中，复购用户（2018年有效订单数 >= 2）的占比百分比（复购率，保留2位小数）。",
        "golden_sql": """
            WITH user_cnt AS (
                SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS cnt
                FROM orders o JOIN customers c ON o.customer_id = c.customer_id
                WHERE o.order_status = 'delivered' AND strftime('%Y', o.order_purchase_timestamp) = '2018'
                GROUP BY c.customer_unique_id
            )
            SELECT ROUND(SUM(CASE WHEN cnt >= 2 THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(customer_unique_id), 2) AS val
            FROM user_cnt;
        """
    },
    {
        "id": "H3", "difficulty": "Hard", "trap_type": "全量历史首单界定老客GMV贡献",
        "question": "在 2018年8月（2018-08）产生有效订单的买家中，历史老客（即在 2018-08-01 之前就有过至少一笔有效订单的独立买家）在 2018年8月 贡献的有效GMV是多少（保留2位小数）？",
        "golden_sql": """
            WITH first_ord AS (
                SELECT c.customer_unique_id, MIN(o.order_purchase_timestamp) AS min_dt
                FROM orders o JOIN customers c ON o.customer_id = c.customer_id
                WHERE o.order_status = 'delivered'
                GROUP BY c.customer_unique_id
            )
            SELECT ROUND(SUM(oi.price), 2) AS val
            FROM orders o
            JOIN order_items oi ON o.order_id = oi.order_id
            JOIN customers c ON o.customer_id = c.customer_id
            JOIN first_ord fo ON c.customer_unique_id = fo.customer_unique_id
            WHERE o.order_status = 'delivered'
              AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08'
              AND fo.min_dt < '2018-08-01';
        """
    },
    {
        "id": "H4", "difficulty": "Hard", "trap_type": "帕累托头部Top3品类集中度占比",
        "question": "在 2018年8月（2018-08）中，有效GMV排名前 3 的英文商品品类，其合计有效GMV占当月大盘总有效GMV的百分比是多少（保留2位小数）？",
        "golden_sql": """
            WITH cat_gmv AS (
                SELECT COALESCE(t.product_category_name_english, 'unknown') AS cat, SUM(oi.price) AS gmv
                FROM orders o
                JOIN order_items oi ON o.order_id = oi.order_id
                LEFT JOIN products p ON oi.product_id = p.product_id
                LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
                WHERE o.order_status = 'delivered' AND strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08'
                GROUP BY 1 ORDER BY gmv DESC
            )
            SELECT ROUND((SELECT SUM(gmv) FROM (SELECT gmv FROM cat_gmv LIMIT 3)) * 100.0 / SUM(gmv), 2) AS val
            FROM cat_gmv;
        """
    },
    {
        "id": "H5", "difficulty": "Hard", "trap_type": "跨月CTE透视与门槛过滤求最高环比增速",
        "question": "对比 2018年7月（2018-07）与 2018年8月（2018-08），在两个月有效GMV均超过 30000 的英文商品品类中，8月相比7月有效GMV环比增长率最高的品类，其环比增长率百分比是多少（保留2位小数）？",
        "golden_sql": """
            WITH m_cat AS (
                SELECT COALESCE(t.product_category_name_english, 'unknown') AS cat,
                       SUM(CASE WHEN strftime('%Y-%m', o.order_purchase_timestamp) = '2018-07' THEN oi.price ELSE 0 END) AS gmv_07,
                       SUM(CASE WHEN strftime('%Y-%m', o.order_purchase_timestamp) = '2018-08' THEN oi.price ELSE 0 END) AS gmv_08
                FROM orders o
                JOIN order_items oi ON o.order_id = oi.order_id
                LEFT JOIN products p ON oi.product_id = p.product_id
                LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
                WHERE o.order_status = 'delivered'
                  AND strftime('%Y-%m', o.order_purchase_timestamp) IN ('2018-07', '2018-08')
                GROUP BY 1
                HAVING gmv_07 > 30000 AND gmv_08 > 30000
            )
            SELECT ROUND((gmv_08 - gmv_07) * 100.0 / gmv_07, 2) AS val
            FROM m_cat ORDER BY val DESC LIMIT 1;
        """
    }
]

# ==============================================================================
# 2. 定义 Baseline 模型（仅提供原始基础表字段名，无业务口径字典 RAG，无自纠错）
# ==============================================================================
def run_naive_baseline(question: str) -> str:
    prompt = f"""
            你是一名资深BI数据分析师，请编写 SQLite 查询语句回答问题：{question}，数据库包含以下表及字段：
            - orders(order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date)
            - order_items(order_id, order_item_id, product_id, price, freight_value)
            - order_payments(order_id, payment_sequential, payment_type, payment_value)
            - customers(customer_id, customer_unique_id, customer_state)
            - products(product_id, product_category_name)
            - category_translation(product_category_name, product_category_name_english)
            - order_reviews(review_id, order_id, review_score)
            要求：只返回最终要求的单个数值列，只输出 ```sql 代码块。
    """
    # resp = llm.invoke(prompt).content
    resp = call_llm_with_rate_limit_guard([HumanMessage(content=prompt)], sleep_seconds=2.0)
    return resp.split("```sql")[-1].split("```")[0].strip() if "```sql" in resp else resp.strip()

def extract_and_match(df: pd.DataFrame, gt_val: float, tol: float = 0.05) -> tuple:
    """智能提取 DataFrame 第一行中的数值并与标准答案比对"""
    if df is None or df.empty:
        return "Empty/Err", False
    numeric_vals = []
    for col in df.columns:
        try:
            v = float(df.iloc[0][col])
            numeric_vals.append(round(v, 2))
            if abs(v - float(gt_val)) <= tol:
                return round(v, 2), True
        except:
            continue
    return (numeric_vals[-1] if numeric_vals else "Non-Num"), False

# ==============================================================================
# 3. 批量执行 15 道分层评测并汇总指标
# ==============================================================================
print("开始运行 15 道分层评测集...\n")
eval_results = []
for item in stratified_benchmark:
    # 1. 计算 Ground Truth 标准答案
    gt_df = pd.read_sql(item["golden_sql"], conn)
    gt_val = round(float(gt_df.iloc[0, 0]), 2)
    
    # 2. 评测 Baseline (无业务口径 RAG)
    try:
        base_sql = run_naive_baseline(item["question"])
        base_df = pd.read_sql(base_sql, conn)
        base_val, base_ok = extract_and_match(base_df, gt_val)
    except Exception as e:
        base_val, base_ok = "SQL_Error", False
        
    # 3. 评测 我们的 BI Agent (口径 RAG + 自纠错闭环)

    agent_out = execute_sql_with_self_correction(
        item["question"] + "（注意：SELECT 子句请只返回最终要求的这 1 个数值列 AS val）",
        max_retries=3, verbose=False
    )
    agent_val, agent_ok = extract_and_match(agent_out["df"], gt_val)
    
    print(f'''[{item['id']} | 
            {item['difficulty']:<6}] {item['trap_type']:<28} | 
            GT: {gt_val:<10} | 
            Base: {str(base_val):<10} ({'✅' if base_ok else '❌'}) | 
            Agent: {str(agent_val):<10} ({'✅' if agent_ok else '❌'} 
            重试:{agent_out['attempts']}次)'''
         )
    
    eval_results.append({
        "题号": item["id"],
        "难度": item["difficulty"],
        "考察业务痛点/陷阱": item["trap_type"],
        "标准答案(GT)": gt_val,
        "Baseline输出": base_val,
        "Baseline正确": 1 if base_ok else 0,
        "Agent输出": agent_val,
        "Agent正确": 1 if agent_ok else 0,
        "Agent尝试次数": agent_out["attempts"]
    })

df_detail = pd.DataFrame(eval_results)

# 4. 计算分难度准确率汇总看板 (Summary Table)
summary_df = df_detail.groupby("难度", sort=False).agg(
    题目数=("题号", "count"),
    Baseline准确率=("Baseline正确", lambda x: f"{x.mean()*100:.1f}%"),
    Agent准确率=("Agent正确", lambda x: f"{x.mean()*100:.1f}%"),
    平均执行次数=("Agent尝试次数", "mean")
).reset_index()

total_row = pd.DataFrame([{
    "难度": "Overall (总计)",
    "题目数": len(df_detail),
    "Baseline准确率": f"{df_detail['Baseline正确'].mean()*100:.1f}%",
    "Agent准确率": f"{df_detail['Agent正确'].mean()*100:.1f}%",
    "平均执行次数": round(df_detail["Agent尝试次数"].mean(), 2)
}])
summary_df = pd.concat([summary_df, total_row], ignore_index=True)

display(Markdown("--- \n ### 分层评测汇总看板"))
display(Markdown(summary_df.to_markdown(index=False)))

display(Markdown("### 15道评测题逐题对比明细表"))
display(Markdown(df_detail.to_markdown(index=False)))

开始运行 15 道分层评测集...

  通道 [Google-Gemini-4-hydrogen] 异常 (Request timed out....) -> 切下一通道 (1/12)
  通道 [Google-Gemini-3.8-Flash] 限流 (429) -> 冷却 35s，秒切下一通道 (2/12)
[E1 | 
            Easy  ] 有效GMV口径(剔除未送达与运费)            | 
            GT: 838576.64  | 
            Base: 996973.51  (❌) | 
            Agent: 838576.64  (✅ 
            重试:1次)
[E2 | 
            Easy  ] 独立买家UV口径(customer_unique_id) | 
            GT: 6310.0     | 
            Base: 6380.0     (❌) | 
            Agent: 6310.0     (✅ 
            重试:1次)
[E3 | 
            Easy  ] 订单取消率口径(全量分母)                | 
            GT: 1.29       | 
            Base: 1.29       (✅) | 
            Agent: 1.29       (✅ 
            重试:1次)
[E4 | 
            Easy  ] 有效件单价ASP口径                   | 
            GT: 124.65     | 
            Base: 124.74     (❌) | 
            Agent: 124.65     (✅ 
            重试:1次)
[E5 | 
            Easy  ] 平均履约时长(julianday语法)          | 
            GT: 8.96       | 
            Base: 8.96       (✅) | 
     

--- 
 ### 分层评测汇总看板

| 难度           |   题目数 | Baseline准确率   | Agent准确率   |   平均执行次数 |
|:---------------|---------:|:-----------------|:--------------|---------------:|
| Easy           |        5 | 40.0%            | 100.0%        |           1    |
| Medium         |        5 | 60.0%            | 80.0%         |           1.8  |
| Hard           |        5 | 20.0%            | 80.0%         |           1.6  |
| Overall (总计) |       15 | 40.0%            | 86.7%         |           1.47 |

### 15道评测题逐题对比明细表

| 题号   | 难度   | 考察业务痛点/陷阱                   |   标准答案(GT) | Baseline输出   |   Baseline正确 | Agent输出   |   Agent正确 |   Agent尝试次数 |
|:-------|:-------|:------------------------------------|---------------:|:---------------|---------------:|:------------|------------:|----------------:|
| E1     | Easy   | 有效GMV口径(剔除未送达与运费)       |      838577    | 996973.51      |              0 | 838576.64   |           1 |               1 |
| E2     | Easy   | 独立买家UV口径(customer_unique_id)  |        6310    | 6380.0         |              0 | 6310.0      |           1 |               1 |
| E3     | Easy   | 订单取消率口径(全量分母)            |           1.29 | 1.29           |              1 | 1.29        |           1 |               1 |
| E4     | Easy   | 有效件单价ASP口径                   |         124.65 | 124.74         |              0 | 124.65      |           1 |               1 |
| E5     | Easy   | 平均履约时长(julianday语法)         |           8.96 | 8.96           |              1 | 8.96        |           1 |               1 |
| M1     | Medium | 客单价AOV分母防膨胀(去重订单数)     |         140.92 | 166.88         |              0 | 140.92      |           1 |               1 |
| M2     | Medium | 订单连带率UPT口径                   |           1.15 | 1.15           |              1 | 1.15        |           1 |               1 |
| M3     | Medium | 四表关联与英文品类Top1有效GMV       |      119391    | 138816.72      |              0 | 119391.01   |           1 |               3 |
| M4     | Medium | 履约延迟率口径                      |           8.24 | 8.24           |              1 | 99.56       |           0 |               3 |
| M5     | Medium | 参评有效订单差评率(订单级去重)      |           9.02 | 9.02           |              1 | 8.99        |           1 |               1 |
| H1     | Hard   | 一对多关联防扇出膨胀(Fan-out Trap)  |      677561    | 809578.71      |              0 | 677560.52   |           1 |               1 |
| H2     | Hard   | CTE二次聚合计算周期用户复购率       |           2.1  | 2.12           |              1 | 2.1         |           1 |               1 |
| H3     | Hard   | 全量历史首单界定老客GMV贡献         |       18517.6  | SQL_Error      |              0 | 18517.56    |           1 |               2 |
| H4     | Hard   | 帕累托头部Top3品类集中度占比        |          29.82 | 29.97          |              0 | 29.82       |           1 |               1 |
| H5     | Hard   | 跨月CTE透视与门槛过滤求最高环比增速 |          15.33 | SQL_Error      |              0 | Empty/Err   |           0 |               3 |